In [10]:
import numpy as np
import pandas as pd
import os
import sys

sys.path.append(os.path.abspath(os.path.join("..")))

import engine
import utils

## Prediction

### Load Human Data

In [11]:
df_human = pd.read_csv("../../../data/human_data/prediction/prediction_long.csv").drop(columns=["Unnamed: 0"])
world_nums = df_human["world"].unique()

### Determine Ground Truth

In [41]:
gt_locs = {}
for world_num in world_nums:
    gt_locs[world_num] = {}
    for hole in [1,2,3]:
        world = utils.load_trial(world_num, experiment="prediction", hole=hole, drop_noise=0.0, col_mean=1.0, col_sd=0.0)
        world["hole_dropped_into"] = hole - 1
        sim = engine.run_simulation(world, convert_coordinates=True)
        outcome_x = sim["ball_position"][-1]["x"]
        gt_locs[world_num][hole] = outcome_x

### Modify df with ground truth

In [42]:
df_human["ground_truth"] = df_human.apply(lambda row: gt_locs[row["world"]][row["hole"]], axis=1)

### Calculate Average Distance

In [52]:
# Function to calculate the average distance
def calculate_average_distance(df):
    return np.mean(np.abs(df["response"] - df["ground_truth"]))

In [54]:
np.round(calculate_average_distance(df_human), 2)

61.72

### Bootstrap Confidence Intervals

In [56]:
# Perform bootstrapping
bootstrap_means = []
for _ in range(1000):
    sample = df_human.sample(frac=1, replace=True)
    bootstrap_means.append(calculate_average_distance(sample))

# Calculate the 95% confidence interval
lower_bound = np.round(np.percentile(bootstrap_means, 2.5), 2)
upper_bound = np.round(np.percentile(bootstrap_means, 97.5), 2)

(lower_bound, upper_bound)

(61.15, 62.35)